# 51 — DeepEval Conversational Metrics

## What you'll learn

- **`ConversationalTestCase`** — wraps a list of `LLMTestCase` turns so the entire conversation trajectory is scored, not just the last reply
- **`KnowledgeRetentionMetric`** — does the bot remember facts shared in earlier turns?
- **`ConversationCompletenessMetric`** — did the bot address every user intent across all turns?
- **`ConversationRelevancyMetric`** — do bot replies stay on-topic throughout the conversation?
- **Stateful vs stateless chatbot comparison** — quantified score difference with the same test script
- **Turn-level vs conversation-level scoring** — why single-turn metrics miss 73% of production chatbot failures

## Context

> Research on production chatbot failures shows that **73% of failures are cross-turn**: the bot "forgets" a fact the user mentioned two turns ago, goes off-topic after a topic switch, or partially addresses a multi-intent message. Standard single-turn `LLMTestCase` metrics catch none of these — you need `ConversationalTestCase` to score the full trajectory.

In [ ]:
%pip install -q deepeval langchain langchain-openai python-dotenv

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

print("API key set.")

In [ ]:
# ── Concept: turn-level eval vs conversation-level eval ──────────────────────
#
# LLMTestCase: scores ONE turn
#   input        = the user's message
#   actual_output = the bot's reply
#
# ConversationalTestCase: scores the FULL TRAJECTORY
#   turns = [LLMTestCase(turn1), LLMTestCase(turn2), ...]
#
# Metrics like KnowledgeRetention look back across all turns to check
# whether the bot remembered facts from earlier in the conversation.

from deepeval.test_case import ConversationalTestCase, LLMTestCase

demo_turns = [
    LLMTestCase(input="My name is Alex.", actual_output="Nice to meet you, Alex!"),
    LLMTestCase(input="What is my name?", actual_output="Your name is Alex."),
]
demo_case = ConversationalTestCase(turns=demo_turns)

print(f"ConversationalTestCase has {len(demo_case.turns)} turns.")
print("Conversation eval scores the full trajectory, not just the last answer.")

In [ ]:
# ── Run stateful and stateless chatbots ──────────────────────────────────────
import sys

sys.path.insert(0, "..")  # make src importable when running from examples/51-*

from src.tools import STATEFUL_CONVERSATION
from src.workflow import run_stateful_chat, run_stateless_chat

print("Running stateful chatbot...")
stateful_history = run_stateful_chat(STATEFUL_CONVERSATION)
print("Running stateless chatbot...")
stateless_history = run_stateless_chat(STATEFUL_CONVERSATION)

# Side-by-side preview of turn 3 ("What's my name?")
# history is flat: [user0, bot0, user1, bot1, ...] so turn 3 is indices 4/5
sf_q = stateful_history[4]["content"]
sf_a = stateful_history[5]["content"]
sl_q = stateless_history[4]["content"]
sl_a = stateless_history[5]["content"]

print("\n[Turn 3 — 'What's my name?']")
print(f"  Stateful  Q: {sf_q}")
print(f"  Stateful  A: {sf_a[:200]}")
print(f"  Stateless Q: {sl_q}")
print(f"  Stateless A: {sl_a[:200]}")

In [ ]:
# ── KnowledgeRetention: stateful vs stateless ─────────────────────────────────
from deepeval.metrics.conversational import KnowledgeRetentionMetric
from deepeval.test_case import ConversationalTestCase, LLMTestCase


def history_to_turns(history: list[dict]) -> list[LLMTestCase]:
    """Convert flat role/content history to a list of LLMTestCase turns."""
    turns = []
    for i in range(0, len(history), 2):
        user_msg = history[i]["content"]
        bot_msg = history[i + 1]["content"] if i + 1 < len(history) else ""
        turns.append(LLMTestCase(input=user_msg, actual_output=bot_msg))
    return turns


stateful_case = ConversationalTestCase(turns=history_to_turns(stateful_history))
stateless_case = ConversationalTestCase(turns=history_to_turns(stateless_history))

kr_metric = KnowledgeRetentionMetric(threshold=0.7, model="gpt-4o-mini")

for label, case in [("Stateful", stateful_case), ("Stateless", stateless_case)]:
    kr_metric.measure(case)
    status = "PASS" if kr_metric.is_successful() else "FAIL"
    print(f"{label} KnowledgeRetention: {kr_metric.score:.2f} | {status}")
    print(f"  Reason: {kr_metric.reason[:200]}\n")

In [ ]:
# ── ConversationRelevancy on the RELEVANCE_CONVERSATION ───────────────────────
from deepeval.metrics.conversational import ConversationRelevancyMetric
from src.tools import RELEVANCE_CONVERSATION

print("Running relevance conversation (stateful only)...")
rel_history = run_stateful_chat(RELEVANCE_CONVERSATION)
rel_case = ConversationalTestCase(turns=history_to_turns(rel_history))

cr_metric = ConversationRelevancyMetric(threshold=0.7, model="gpt-4o-mini")
cr_metric.measure(rel_case)
status = "PASS" if cr_metric.is_successful() else "FAIL"
print(f"ConversationRelevancy: {cr_metric.score:.2f} | {status}")
print(f"  Reason: {cr_metric.reason[:200]}\n")

# ── Summary table: stateful vs stateless ──────────────────────────────────────
print("\n=== Summary ===")
print(f"{'Metric':<30} {'Stateful':>10} {'Stateless':>10}")
print("-" * 52)

kr_metric.measure(stateful_case)
sf_kr = kr_metric.score
kr_metric.measure(stateless_case)
sl_kr = kr_metric.score

cr_metric.measure(stateful_case)
sf_cr = cr_metric.score
cr_metric.measure(stateless_case)
sl_cr = cr_metric.score

print(f"{'KnowledgeRetention':<30} {sf_kr:>10.2f} {sl_kr:>10.2f}")
print(f"{'ConversationRelevancy':<30} {sf_cr:>10.2f} {sl_cr:>10.2f}")

## Exercises

1. **Drop relevancy** — insert an off-topic bot reply in turn 4 of the RELEVANCE_CONVERSATION (e.g., the bot talks about cooking instead of the GIL). Re-run `ConversationRelevancyMetric` and watch the score drop.

2. **Perfect retention** — build a 6-turn conversation where the bot correctly recalls every fact the user shares. Verify that `KnowledgeRetentionMetric` stays above 0.9.

3. **Stateless vs stateful task** — design a 5-turn task that *requires* memory to complete correctly (e.g., the user sets a goal in turn 1 and asks for progress in turn 5). Compare the two chatbot scores.

4. **Perfect score design** — what does a conversation look like that achieves `KnowledgeRetention = 1.0`? Write it as a `ConversationalTestCase` with hard-coded outputs and verify the score.

---

## What's next

- `48-deepeval-hallucination-bias` — detect hallucination and demographic bias in generated text
- `50-deepeval-agentic-eval` — score multi-step agent tool-use trajectories
- `52-deepeval-synthesizer` — auto-generate diverse test datasets from a knowledge base